# **EXPLORATORY DATA ANALYSIS**

In [26]:
import pandas as pd 
import numpy as np

# **RESULTS**

In [97]:
df_res = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\bronze\results\all_seasons_results.parquet")

In [98]:
df_res.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,grid_position,...,race_time_text,fastest_lap_number,fastest_lap_time,fastest_lap_rank,fastest_lap_speed,fastest_lap_speed_units,is_winner,is_podium,is_points,is_dnf
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,...,1:29:33.283,53.0,1:26.469,4.0,220.782,kph,True,True,True,False
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,...,+5.036,50.0,1:26.444,3.0,220.845,kph,False,True,True,False
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,...,+6.309,57.0,1:26.373,2.0,221.027,kph,False,True,True,False
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,8,...,+7.069,54.0,1:25.945,1.0,222.128,kph,False,False,True,False
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,ALO,14,mclaren,10,...,+27.886,57.0,1:26.978,7.0,219.489,kph,False,False,True,False


In [99]:
df_res.shape

(3458, 26)

In [100]:
df_res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3458 entries, 0 to 3457
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   season                   3458 non-null   int64  
 1   round_number             3458 non-null   int64  
 2   race_name                3458 non-null   object 
 3   circuit_ref              3458 non-null   object 
 4   race_date                3458 non-null   object 
 5   driver_ref               3458 non-null   object 
 6   driver_code              3458 non-null   object 
 7   driver_number            3458 non-null   int64  
 8   constructor_ref          3458 non-null   object 
 9   grid_position            3458 non-null   int64  
 10  finish_position          3458 non-null   int64  
 11  position_text            3458 non-null   object 
 12  points                   3458 non-null   float64
 13  laps_completed           3458 non-null   int64  
 14  status                  

As there is no **primary key** present therefore we create **candifate key df_ref** which will act as this table's **primary key** and checking whether combo of driver_ref, season, round_no has any duplicates or not

In [101]:
pk_test = df_res.duplicated(
    subset=["driver_ref", "season", "round_number"]
).sum()
print(f"Duplicates with driver_ref + season + round_number: {pk_test}")

Duplicates with driver_ref + season + round_number: 0


In [102]:
df_res["results_ref"] = df_res["season"].astype(str).str[2:4] + "_" + df_res["round_number"].astype(str) + "_" + df_res["driver_ref"].astype(str)

In [103]:
df_res["race_date"]    = pd.to_datetime(df_res["race_date"])
df_res["fastest_lap_number"] = df_res["fastest_lap_number"].astype(pd.Int64Dtype())
df_res["fastest_lap_rank"] = df_res["fastest_lap_rank"].astype(pd.Int64Dtype())
df_res["fastest_lap_speed"]=df_res["fastest_lap_speed"].astype("float64")

* Dropping **race_time_milis** and **race_time_text** as it already exists in laps data
* Dropping **fastest_lap_speed_units** as we have renamed the column **fastest_lap_speed** to **fastest_lap_speed_kph** 

In [104]:
df_res.drop(columns={"fastest_lap_speed_units","race_time_millis","race_time_text"},inplace=True)

* Using existing laps data, we are filling the null values of **fastest_lap_time** and **fastest_lap_number**
* Those with **0 laps**, their lap time and number is considered **0**

In [105]:
import pandas as pd

df_laps    = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\bronze\laps\all_seasons_laps.parquet")

JOIN_KEYS = ["season", "round_number", "driver_ref"]

valid_laps = df_laps.dropna(subset=["lap_time_ms"]).copy()

fastest = (
    valid_laps
    .sort_values("lap_time_ms")
    .drop_duplicates(subset=JOIN_KEYS, keep="first")
    [JOIN_KEYS + ["lap_number", "lap_time"]]
    .rename(columns={
        "lap_number": "fl_number_from_laps",
        "lap_time":   "fl_time_from_laps",
    })
)

df_res = df_res.merge(fastest, on=JOIN_KEYS, how="left")

df_res["fastest_lap_number"] = df_res["fastest_lap_number"].fillna(
    df_res["fl_number_from_laps"]
)

df_res["fastest_lap_time"] = df_res["fastest_lap_time"].fillna(
    df_res["fl_time_from_laps"]
)

df_res.drop(columns=["fl_number_from_laps", "fl_time_from_laps"], inplace=True)

zero_laps_mask = df_res["laps_completed"] == 0

df_res.loc[zero_laps_mask, "fastest_lap_number"] = df_res.loc[
    zero_laps_mask, "fastest_lap_number"
].fillna(0)

df_res.loc[zero_laps_mask, "fastest_lap_time"] = df_res.loc[
    zero_laps_mask, "fastest_lap_time"
].fillna("0.0")

print("=== fastest_lap_number null count after fill ===")
print(df_res["fastest_lap_number"].isnull().sum())

print("\n=== fastest_lap_time null count after fill ===")
print(df_res["fastest_lap_time"].isnull().sum())

print("\n=== Sample rows with 0 laps completed ===")
print(
    df_res[df_res["laps_completed"] == 0][
        ["season", "round_number", "driver_ref", "laps_completed",
         "fastest_lap_number", "fastest_lap_time"]
    ].head(10).to_string(index=False)
)

print("\n=== Sample rows where null was filled from laps ===")
filled_sample = df_res[
    df_res["fastest_lap_time"].notna() & (df_res["laps_completed"] > 0)
][["season", "round_number", "driver_ref", "laps_completed",
   "fastest_lap_number", "fastest_lap_time"]].head(10)
print(filled_sample.to_string(index=False))

=== fastest_lap_number null count after fill ===
0

=== fastest_lap_time null count after fill ===
0

=== Sample rows with 0 laps completed ===
 season  round_number      driver_ref  laps_completed  fastest_lap_number fastest_lap_time
   2018             4            ocon               0                   0              0.0
   2018             4        sirotkin               0                   0              0.0
   2018             5        grosjean               0                   0              0.0
   2018             5           gasly               0                   0              0.0
   2018             5      hulkenberg               0                   0              0.0
   2018             7 brendon_hartley               0                   0              0.0
   2018             7          stroll               0                   0              0.0
   2018             8            ocon               0                   0              0.0
   2018             8           gasly

Dropping record of **Bortoleto on 23rd November, 2025 at Las Vegas GP** due to unavailability of data

In [107]:
row = df_res[
    (df_res["driver_ref"]   == "bortoleto") &
    (df_res["season"]       == 2025)        &
    (df_res["round_number"] == 22)
]
df_res = df_res.drop(index=row.index).reset_index(drop=True)

* Here we are filling the null values of **fastest_lap_speed** by first getting the distance of circuits of every venue throught the years 
* We already have the fastest time for every driver and perfrorm simple calculation of **Speed = Distance/Time**

In [108]:
import re
# ── CIRCUIT DISTANCES (km) ────────────────────────────────────────────────────
CIRCUIT_DISTANCES = {
    "albert_park": {2018: 5.303, 2019: 5.303, 2020: 5.303, 2021: 5.303, 2022: 5.278, 2023: 5.278, 2024: 5.278, 2025: 5.278},
    "bahrain":     {2018: 5.412, 2019: 5.412, 2020: 5.412, 2021: 5.412, 2022: 5.412, 2023: 5.412, 2024: 5.412, 2025: 5.412},
    "shanghai":    {2018: 5.451, 2019: 5.451, 2024: 5.451, 2025: 5.451},
    "baku":        {2018: 6.003, 2019: 6.003, 2020: 6.003, 2021: 6.003, 2022: 6.003, 2023: 6.003, 2024: 6.003, 2025: 6.003},
    "catalunya":   {2018: 4.655, 2019: 4.655, 2020: 4.675, 2021: 4.657, 2022: 4.657, 2023: 4.657, 2024: 4.657, 2025: 4.657},
    "monaco":      {2018: 3.337, 2019: 3.337, 2020: 3.337, 2021: 3.337, 2022: 3.337, 2023: 3.337, 2024: 3.337, 2025: 3.337},
    "villeneuve":  {2018: 4.361, 2019: 4.361, 2020: 4.361, 2021: 4.361, 2022: 4.361, 2023: 4.361, 2024: 4.361, 2025: 4.361},
    "paul_ricard": {2018: 5.842, 2019: 5.842, 2020: 5.842, 2021: 5.842, 2022: 5.842},
    "red_bull_ring":{2018: 4.318, 2019: 4.318, 2020: 4.318, 2021: 4.318, 2022: 4.318, 2023: 4.318, 2024: 4.318, 2025: 4.318},
    "silverstone": {2018: 5.891, 2019: 5.891, 2020: 5.891, 2021: 5.891, 2022: 5.891, 2023: 5.891, 2024: 5.891, 2025: 5.891},
    "hockenheimring":  {2018: 4.574, 2019: 4.574},
    "hungaroring": {2018: 4.381, 2019: 4.381, 2020: 4.381, 2021: 4.381, 2022: 4.381, 2023: 4.381, 2024: 4.381, 2025: 4.381},
    "spa":         {2018: 7.004, 2019: 7.004, 2020: 7.004, 2021: 7.004, 2022: 7.004, 2023: 7.004, 2024: 7.004, 2025: 7.004},
    "zandvoort":   {2021: 4.259, 2022: 4.259, 2023: 4.259, 2024: 4.259, 2025: 4.259},
    "monza":       {2018: 5.793, 2019: 5.793, 2020: 5.793, 2021: 5.793, 2022: 5.793, 2023: 5.793, 2024: 5.793, 2025: 5.793},
    "marina_bay":  {2018: 5.063, 2019: 5.063, 2020: 5.063, 2021: 5.063, 2022: 5.063, 2023: 4.940, 2024: 4.940, 2025: 4.940},
    "sochi":       {2018: 5.848, 2019: 5.848, 2020: 5.848, 2021: 5.848},
    "suzuka":      {2018: 5.807, 2019: 5.807, 2020: 5.807, 2021: 5.807, 2022: 5.807, 2023: 5.807, 2024: 5.807, 2025: 5.807},
    "losail":      {2021: 5.380, 2023: 5.380, 2024: 5.380, 2025: 5.380},
    "americas":    {2018: 5.513, 2019: 5.513, 2020: 5.513, 2021: 5.513, 2022: 5.513, 2023: 5.513, 2024: 5.513, 2025: 5.513},
    "rodriguez":   {2018: 4.304, 2019: 4.304, 2020: 4.304, 2021: 4.304, 2022: 4.304, 2023: 4.304, 2024: 4.304, 2025: 4.304},
    "interlagos":  {2018: 4.309, 2019: 4.309, 2020: 4.309, 2021: 4.309, 2022: 4.309, 2023: 4.309, 2024: 4.309, 2025: 4.309},
    "yas_marina":  {2018: 5.554, 2019: 5.554, 2020: 5.554, 2021: 5.281, 2022: 5.281, 2023: 5.281, 2024: 5.281, 2025: 5.281},
    "imola":       {2020: 4.909, 2021: 4.909, 2022: 4.909, 2023: 4.909, 2024: 4.909, 2025: 4.909},
    "portimao":    {2020: 4.684, 2021: 4.684},
    "istanbul":    {2020: 5.338, 2021: 5.338},
    "mugello":     {2020: 5.245},
    "jeddah":      {2021: 6.174, 2022: 6.174, 2023: 6.174, 2024: 6.174, 2025: 6.174},
    "miami":       {2022: 5.412, 2023: 5.412, 2024: 5.412, 2025: 5.412},
    "vegas":       {2023: 6.201, 2024: 6.201, 2025: 6.201},
}

SEASON_ROUND_OVERRIDES = {
    (2020, 16): ("bahrain_outer", 3.543),
}

# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────

def parse_lap_time_to_ms(time_str):
    """Convert lap time string to milliseconds"""
    if pd.isna(time_str) or str(time_str).strip() in ("0.0", "0", "None", "N/A", ""):
        return None
    time_str = str(time_str).strip()
    m = re.fullmatch(r"(\d+):(\d{2})\.(\d+)", time_str)
    if m:
        minutes = int(m.group(1))
        seconds = int(m.group(2))
        millis  = int(m.group(3).ljust(3, "0")[:3])
        return (minutes * 60 + seconds) * 1000 + millis
    m = re.fullmatch(r"(\d+)\.(\d+)", time_str)
    if m:
        seconds = int(m.group(1))
        millis  = int(m.group(2).ljust(3, "0")[:3])
        return seconds * 1000 + millis
    return None

def get_circuit_distance(circuit_ref, season, round_number=None):
    """Look up circuit distance in km"""
    if round_number is not None:
        key = (int(season), int(round_number))
        if key in SEASON_ROUND_OVERRIDES:
            _, dist = SEASON_ROUND_OVERRIDES[key]
            return dist
    seasons = CIRCUIT_DISTANCES.get(circuit_ref)
    if seasons is None:
        return None
    return seasons.get(int(season))

# ── FILL NULL fastest_lap_speed ONLY ─────────────────────────────────────────

def calculate_speed(row):
    # 0 laps → speed is 0.0
    if row["laps_completed"] == 0:
        return 0.0

    # Parse lap time to ms
    lap_time_ms = parse_lap_time_to_ms(row["fastest_lap_time"])
    if lap_time_ms is None or lap_time_ms <= 0:
        return None

    # Get circuit distance
    distance_km = get_circuit_distance(
        row["circuit_ref"], row["season"], row["round_number"]
    )
    if distance_km is None:
        return None

    # speed = distance / time
    time_hours = lap_time_ms / 3_600_000
    return round(distance_km / time_hours, 3)

# ── Apply only to NULL rows ───────────────────────────────────────────────────
null_mask = df_res["fastest_lap_speed"].isna()
df_res.loc[null_mask, "fastest_lap_speed"] = df_res[null_mask].apply(
    calculate_speed, axis=1
)

Rectifying the **fastest_lap_rank** column based on the new data we fastest_lap_time we have manipulated

In [109]:
# we are ranking the drivers by fastest time for every lap in each round
df_res["_fl_time_ms"] = df_res["fastest_lap_time"].map(parse_lap_time_to_ms)

# Drivers with 0 laps or null time get rank 0
df_res["fastest_lap_rank"] = df_res.groupby(
    ["season", "round_number"]
)["_fl_time_ms"].rank(method="min", ascending=True).astype("Int16")
invalid_mask = (
    (df_res["laps_completed"] == 0) |
    (df_res["_fl_time_ms"].isna())
)
df_res.loc[invalid_mask, "fastest_lap_rank"] = 0
df_res = df_res.drop(columns=["_fl_time_ms"])

* On **29th August, 2021, Belgian Grand Prix**, a unique happened players were distributed fraction or half the points according to their standings due to **occurance of torrential rain** 
* Therefore flagging it with "Finished_torrential_rain"

In [110]:
mask = (
    (df_res["season"]       == 2021) &
    (df_res["round_number"]    == 12) &
    (df_res["status"]       == "Finished")
)

df_res.loc[mask, "status"] = "Finished_torrential_rain"

In [111]:
df_res.rename(columns={"fastest_lap_speed": "fastest_lap_speed_kph"}, inplace=True)

In [112]:
df_res.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,grid_position,...,status,fastest_lap_number,fastest_lap_time,fastest_lap_rank,fastest_lap_speed_kph,is_winner,is_podium,is_points,is_dnf,results_ref
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,...,Finished,53,1:26.469,4,220.782,True,True,True,False,18_1_vettel
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,...,Finished,50,1:26.444,3,220.845,False,True,True,False,18_1_hamilton
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,...,Finished,57,1:26.373,2,221.027,False,True,True,False,18_1_raikkonen
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,8,...,Finished,54,1:25.945,1,222.128,False,False,True,False,18_1_ricciardo
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,ALO,14,mclaren,10,...,Finished,57,1:26.978,7,219.489,False,False,True,False,18_1_alonso


In [113]:
df_res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3457 entries, 0 to 3456
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   season                 3457 non-null   int64         
 1   round_number           3457 non-null   int64         
 2   race_name              3457 non-null   object        
 3   circuit_ref            3457 non-null   object        
 4   race_date              3457 non-null   datetime64[ns]
 5   driver_ref             3457 non-null   object        
 6   driver_code            3457 non-null   object        
 7   driver_number          3457 non-null   int64         
 8   constructor_ref        3457 non-null   object        
 9   grid_position          3457 non-null   int64         
 10  finish_position        3457 non-null   int64         
 11  position_text          3457 non-null   object        
 12  points                 3457 non-null   float64       
 13  lap

In [114]:
df_res.to_parquet(r"C:\Users\Asus\Desktop\Formula1\data\silver\results\cleaned_results.parquet")

In [115]:
df = pd.read_parquet(r"C:\Users\Asus\Desktop\Formula1\data\silver\results\cleaned_results.parquet")
df.head()

,season,round_number,race_name,circuit_ref,race_date,driver_ref,driver_code,driver_number,constructor_ref,grid_position,...,status,fastest_lap_number,fastest_lap_time,fastest_lap_rank,fastest_lap_speed_kph,is_winner,is_podium,is_points,is_dnf,results_ref
0,2018,1,Australian Grand Prix,albert_park,2018-03-25,vettel,VET,5,ferrari,3,...,Finished,53,1:26.469,4,220.782,True,True,True,False,18_1_vettel
1,2018,1,Australian Grand Prix,albert_park,2018-03-25,hamilton,HAM,44,mercedes,1,...,Finished,50,1:26.444,3,220.845,False,True,True,False,18_1_hamilton
2,2018,1,Australian Grand Prix,albert_park,2018-03-25,raikkonen,RAI,7,ferrari,2,...,Finished,57,1:26.373,2,221.027,False,True,True,False,18_1_raikkonen
3,2018,1,Australian Grand Prix,albert_park,2018-03-25,ricciardo,RIC,3,red_bull,8,...,Finished,54,1:25.945,1,222.128,False,False,True,False,18_1_ricciardo
4,2018,1,Australian Grand Prix,albert_park,2018-03-25,alonso,ALO,14,mclaren,10,...,Finished,57,1:26.978,7,219.489,False,False,True,False,18_1_alonso
